# Lab 04 challenge: worked solution

Solutions to the two challenge tasks of
[Lab 04: How to remember a rep across conversations, and forget a customer](../04_long_term_memory_and_privacy.ipynb).
Run the lab first; this notebook repeats only the pieces each task needs.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

The pieces both tasks share, copied from the lab: the prompt file, `get_policy`, the Czech check and the questions.

In [3]:
import re
from dataclasses import dataclass
from typing import Literal
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from shared.copilot import load_prompt
from shared.settings import chat_model

V1 = load_prompt("copilot_system_v1.md")
CORPUS = ROOT / "data" / "corpus"
POLICIES = sorted(path.stem.removeprefix("policy_") for path in CORPUS.glob("policy_*.md"))

def get_policy(name: str) -> str:
    """Return the full text of one Chinook support policy.

    Call it before answering any question about shop rules.

    Args:
        name: Policy name, e.g. "refunds" or "escalations".
    """
    path = CORPUS / f"policy_{name}.md"
    if not path.exists():
        return f"No policy named {name!r}. Known: {', '.join(POLICIES)}."
    return path.read_text(encoding="utf-8")

policy_tool = tool(get_policy, parse_docstring=True)

def looks_czech(reply: str) -> bool:
    return bool(re.search(r"[ěščřžůťď]|Dobrý", reply))

CUSTOMER_CASE = "Customer Helena Holý wants a new download link. Remember that she prefers replies in Czech."
REFUND = "A customer asks how long a refund takes. Draft the reply."
PAYPAL = "A customer asks whether we accept PayPal. Draft the reply."

def run_turn(agent, context, thread_id: str, text: str):
    out = agent.invoke({"messages": [{"role": "user", "content": text}]},
                       {"configurable": {"thread_id": thread_id}}, context=context)
    calls = [call["name"] for message in out["messages"] if message.type == "ai" for call in message.tool_calls]
    return calls, out["messages"][-1].text

## Task 1: Put the customer's wish where it belongs

**Step 1.** The context now holds the customer of the case as well as the rep, and `customer_namespace` builds the
customer's namespace from it, as the rep's is built from `rep_id`. **Step 2** follows in the same cell. A second save tool writes under
`("customer", customer_id)`, and `get_preferences` returns both namespaces. Both tools still build the namespace from
the context: the model chooses *the rep* or *this customer* by choosing a tool, never another person.

One choice the task does not ask for: the rep's `save_preference` here no longer offers `language`. A language
then can only be saved as the customer's wish, so the Part D mistake cannot happen through this tool. The cost: a
rep can no longer save their own language for replies. Keep the key if your reps need it, and rely on the prompt
lines and a check of the store instead.

In [4]:
@dataclass
class CaseContext:
    rep_id: str
    customer_id: str

def customer_namespace(context: CaseContext) -> tuple:
    """The store namespace for this case's customer."""
    return ("customer", context.customer_id)


@tool(parse_docstring=True)
def save_preference(key: Literal["reply_format", "sign_off"], value: str, runtime: ToolRuntime[CaseContext]) -> str:
    """Save one of the rep's own preferences for how they write customer replies.

    Only the rep's own style. Never a customer's wish: use save_customer_preference for that.

    Args:
        key: Which preference: reply_format or sign_off.
        value: The rep's preference, in their words.
    """
    runtime.store.put(("rep", runtime.context.rep_id), key, {"value": value})
    return f"Saved for the rep: {key} = {value}"

@tool(parse_docstring=True)
def save_customer_preference(key: Literal["language", "contact_time"], value: str,
                             runtime: ToolRuntime[CaseContext]) -> str:
    """Save a wish of the customer in this case, such as the language they want replies in.

    Args:
        key: Which wish: language or contact_time.
        value: The customer's wish.
    """
    runtime.store.put(customer_namespace(runtime.context), key, {"value": value})
    return f"Saved for this customer: {key} = {value}"

@tool
def get_preferences(runtime: ToolRuntime[CaseContext]) -> str:
    """Return the rep's saved preferences and this customer's saved wishes for customer replies."""
    sections = []
    for title, namespace in (("Rep", ("rep", runtime.context.rep_id)),
                             ("This customer", customer_namespace(runtime.context))):
        rows = [f"- {item.key}: {item.value['value']}" for item in runtime.store.search(namespace)]
        sections.append(f"{title}:\n" + ("\n".join(rows) or "- none saved"))
    return "\n".join(sections)

print(customer_namespace(CaseContext("rep-42", "cust-helena")))

MEMORY = """

# Memory
The rep's preferences and this customer's wishes are not in this prompt: call get_preferences to read them before every suggested reply, also when you call get_policy, and apply both.
When the rep asks you to remember the rep's own style, call save_preference. When it is a wish of the customer in this case, call save_customer_preference."""

('customer', 'cust-helena')


**Step 3.** Three runs, a new store each time: Helena's case, then a refund question in a new thread for Helena, and one for
another customer, Puja, of the same rep.

In [5]:
helena, puja = CaseContext("rep-42", "cust-helena"), CaseContext("rep-42", "cust-puja")
for run in range(1, 4):
    store = InMemoryStore()
    agent = create_agent(chat_model(settings), tools=[policy_tool, save_preference, save_customer_preference,
                                                       get_preferences],
                         system_prompt=V1 + MEMORY, context_schema=CaseContext,
                         checkpointer=InMemorySaver(), store=store)
    calls, _ = run_turn(agent, helena, f"case-helena-{run}", CUSTOMER_CASE)
    _, again = run_turn(agent, helena, f"case-helena-{run}b", REFUND)
    _, other = run_turn(agent, puja, f"case-puja-{run}", REFUND)
    saved = [(item.namespace, item.key, item.value["value"]) for item in store.search(())]
    print(f"run {run}  tools {calls}  saved {saved}")
    print(f"       Helena's next thread in Czech: {looks_czech(again)}  Puja's thread in Czech: {looks_czech(other)}")

run 1  tools ['get_preferences', 'save_customer_preference', 'get_policy']  saved [(('customer', 'cust-helena'), 'language', 'Czech')]
       Helena's next thread in Czech: True  Puja's thread in Czech: False


run 2  tools ['get_preferences', 'save_customer_preference', 'get_policy']  saved [(('customer', 'cust-helena'), 'language', 'Czech')]
       Helena's next thread in Czech: True  Puja's thread in Czech: False


run 3  tools ['get_preferences', 'save_customer_preference', 'get_policy']  saved [(('customer', 'cust-helena'), 'language', 'Czech')]
       Helena's next thread in Czech: True  Puja's thread in Czech: False


In 3 of 3 runs the copilot read the preferences, chose `save_customer_preference`, and *Czech* landed under `('customer', 'cust-helena')`, never in the rep's namespace. Helena's own next thread was in Czech in 3 of 3, and Puja's never (0 of 3): the wish follows the customer, not the rep. The section in `get_preferences` labels whose preferences are whose, and the Memory section says to apply both.

Deleting Helena's data now also means `store.delete` on every key under `("customer", "cust-helena")`, which is
easier than tracing source threads: the namespace itself says whose data it is.

## Task 2: Let the rep forget

It starts from a new store that holds only Part D's mistake, so it does not depend on task 1.

A third tool deletes one of the signed-in rep's memories through `runtime.store`, and one line in the prompt says
when to use it. The store starts with Helena's *Czech* saved as the rep's preference, as in the lab's Part D.

In [6]:
@dataclass
class Context:
    rep_id: str

@tool(parse_docstring=True)
def save_rep_preference(key: Literal["reply_format", "sign_off", "language"], value: str,
                        runtime: ToolRuntime[Context]) -> str:
    """Save one of this rep's preferences for how customer replies are written.

    Call it once for each preference the rep states.

    Args:
        key: Which preference: reply_format, sign_off or language.
        value: The rep's preference, in their words.
    """
    runtime.store.put(("rep", runtime.context.rep_id), key, {"value": value})
    return f"Saved {key} = {value}"
save_rep_preference.name = "save_preference"

@tool
def get_rep_preferences(runtime: ToolRuntime[Context]) -> str:
    """Return this rep's saved preferences for how customer replies are written."""
    items = runtime.store.search(("rep", runtime.context.rep_id))
    return "\n".join(f"{item.key}: {item.value['value']}" for item in items) or "No saved preferences."
get_rep_preferences.name = "get_preferences"

@tool(parse_docstring=True)
def forget_preference(key: Literal["reply_format", "sign_off", "language"], runtime: ToolRuntime[Context]) -> str:
    """Delete one of this rep's saved preferences.

    Args:
        key: Which preference to delete: reply_format, sign_off or language.
    """
    namespace = ("rep", runtime.context.rep_id)
    if runtime.store.get(namespace, key) is None:
        return f"No saved {key} to delete."
    runtime.store.delete(namespace, key)
    return f"Deleted {key}"

REP_MEMORY = """

# Memory
This rep may have saved preferences for how customer replies are written. They are not in this prompt: call get_preferences to read them before every suggested reply, also when you call get_policy.
When the rep asks you to remember something for future replies, call save_preference. When the rep asks you to forget one, call forget_preference."""

FORGET = "Forget the Czech preference, it was for one customer."
for run in range(1, 4):
    store = InMemoryStore()
    store.put(("rep", "rep-42"), "language", {"value": "Czech"})
    agent = create_agent(chat_model(settings), tools=[policy_tool, save_rep_preference, get_rep_preferences,
                                                       forget_preference],
                         system_prompt=V1 + REP_MEMORY, context_schema=Context,
                         checkpointer=InMemorySaver(), store=store)
    calls, reply = run_turn(agent, Context("rep-42"), f"forget-{run}", FORGET)
    left = {item.key: item.value["value"] for item in store.search(("rep", "rep-42"))}
    _, paypal = run_turn(agent, Context("rep-42"), f"paypal-{run}", PAYPAL)
    print(f"run {run}  tools {calls}  left {left}  PayPal reply in Czech: {looks_czech(paypal)}")
print("\nlast reply to the rep:", reply)

run 1  tools ['forget_preference']  left {}  PayPal reply in Czech: False


run 2  tools ['forget_preference']  left {}  PayPal reply in Czech: False


run 3  tools ['forget_preference']  left {}  PayPal reply in Czech: False

last reply to the rep: I've removed the Czech language preference.


In 3 of 3 runs the copilot called `forget_preference` and nothing else, rep 42's namespace was left empty, and the PayPal reply in a new thread was in English. The tool's reply is what the rep sees: *I've removed the Czech language preference.*

A forget tool is the rep's side of Part E. It deletes only the signed-in rep's memories, because the namespace comes
from the context; deleting a customer's data stays a job for your own code, run on a request you can check.